# Matching Names and ORCIDs
Given a list of author names we can search for ORCID metadata using getORCIDS.py. This retrieves data from ORCID profiles including: other names, employment, education, works, peer-reviews, funding and email. In order for these retrievals to work, the ORCID profile data must 1) exist in the profile and 2) be open to the public. If these data are available, it is not unusual for there to be multiple matches to many names, after all, that is why ORCIDs exist.

This notebook explores a possible rubric fort describing the goodness-of-fit of a match between a name and an ORCID. It is, of course, not fool proof, but it may be helpful.

In [3]:
import pandas as pd
from   tabulate import tabulate                     # makes pretty tables in many formats from dataframes
from   IPython.display import display, Markdown     # allows computation results to be displayed in markdown
import  ipywidgets as widgets                        # interactive widgets

## Read paper index and ORCID metadata from git repository (collab) or from files

In [4]:
#
# Data and metadata related to these papers are in a public git repository
# https://github.com/tedhabermann/MooreaStudentPapers
# The index of the papers (a spreadsheet) is in this git repository as a csv file
# read the index of student papers from git repository
#
url = 'https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/index_to_moorea_class_projects_1992-2022.csv'
index_df = pd.read_csv(url)
#
# Read all sheets from the ORCID metadata files in the git repository
# These are csv files spilt out from the ORCID metadata spreadsheet (xlsx) that is in the git repository
#
orcid_metadata_l = ['counts', 'employment', 'education', 'works', 'funding', 'email']
orcid_dfs = {}
for sheet in orcid_metadata_l:
    url = f"https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/orcidMetadata_{sheet}__20260303_14.csv"
    print(f"Reading {sheet} sheet from ORCID metadata file at {url}")
    orcid_dfs[sheet] = pd.read_csv(url)
    print(f"{len(orcid_dfs[sheet])} rows in the {sheet} dataframe")

# Display the dataframes to verify they were read correctly
print(f"{len(index_df)} rows in the index dataframe")           # Display the index count
print(f"Sheets found: {list(orcid_dfs.keys())}")

546 rows in the index dataframe
Reading counts sheet from ORCID metadata file at https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/orcidMetadata_counts__20260303_14.csv
2172 rows in the counts dataframe
Reading employment sheet from ORCID metadata file at https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/orcidMetadata_employment__20260303_14.csv
2582 rows in the employment dataframe
Reading education sheet from ORCID metadata file at https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/orcidMetadata_education__20260303_14.csv
2531 rows in the education dataframe
Reading works sheet from ORCID metadata file at https://raw.githubusercontent.com/tedhabermann/MooreaStudentPapers/refs/heads/main/data/orcidMetadata_works__20260303_14.csv
24957 rows in the works dataframe
Reading funding sheet from ORCID metadata file at https://raw.githubusercontent.com/tedhabermann/MooreaStudentPap

In [ ]:
# data for this analysis are in local files (not for use in collab)
# Read the index of student papers, which includes author names and publication years, into a dataframe
index_file = '/Users/metadatagamechanger/ProjectsAndPlans/FieldStations/MooreaStudentPapers/index_to_moorea_class_projects_1992-2022.xlsx'
index_df = pd.read_excel(index_file, sheet_name='MooreaClass-Export.txt', engine='openpyxl')

# Display the sheet names
print(f"{len(index_df)} rows in the index dataframe")
#
# Read all sheets from the ORCID metadata file (xlsx) into a dictionary of dataframes
#
orcid_metadata_file = '/Users/metadatagamechanger/Repositories/MooreaStudentPapers/data/orcidMetadata__20260303_14.xlsx'
orcid_dfs = pd.read_excel(orcid_metadata_file, sheet_name=None)

# Display the sheet names
print(f"Sheets found: {list(orcid_dfs.keys())}")

## Match Counts
An ORCID search for a name can return any number of matches, i.e. one for each person with that name that has an ORCID. The difficulty of picking the right ORCID increases as the number of matches increases. The counts dataframe in the ORCID output has the input names in a column named 'input'. The number of times these mnames occur is the number of matches for the name. Add these counts to the counts dataframe as the matchCounts column.

In [14]:
#
# Create a series of counts of the number of matches for each input name, and add it to the dataframe
#
counts_df                = orcid_dfs['counts']                                                                                 # get the counts dataframe from the dictionary of dataframes
matchCounts              = counts_df[counts_df['orcid'] != 'Not Found']['input'].value_counts()     # a series with the number of matches for each input name
counts_df['matchCounts'] = counts_df['input'].map(matchCounts).fillna(0).astype(int)                # add the match counts to the dataframe, filling in 0 for names with no matches
counts_df.fillna('', inplace=True)                                                                  # replace NaN with empty string for better display
#counts_df.head(20)                                                                                  # display the first 20 rows of the dataframe with match counts
print(f"{len(counts_df)} rows in the counts dataframe with match counts added")

2172 rows in the counts dataframe with match counts added


In [39]:
studyYear = 1994                        # pick a year to examine


In [40]:
studyYear_df = index_df[index_df['Pub Year'] == studyYear]
author_l = list(studyYear_df['Authors, Primary'].values)                               # display the keys of the 'Authors, Primary' column for the selected year
authorORCIDMetadata_df = counts_df[counts_df['input'].isin(author_l)]         # get the dataframe for the 'authorORCIDMetadata' sheet
#display(Markdown(tabulate(authorORCIDMetadata_df.matchCounts.value_counts().reset_index(), headers=['Match Count', 'Frequency'], tablefmt='pipe', floatfmt='.0f', showindex=False)))
matchCountLimit = 10
df = authorORCIDMetadata_df[authorORCIDMetadata_df['matchCounts'] <= matchCountLimit]
display(Markdown(f'**{len(studyYear_df)} authors in {studyYear} with matches <= {matchCountLimit}**'))
display(Markdown(tabulate(df.sort_values(by='matchCounts'), headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))


**15 authors in 1994 with matches <= 10**

| input          | given                          | family    | orcid               | other-names   | employment   | education   | works   | peer-reviews   | funding   | emails   | other_names   |   Total | journal                                |   matchCounts |
|:---------------|:-------------------------------|:----------|:--------------------|:--------------|:-------------|:------------|:--------|:---------------|:----------|:---------|:--------------|--------:|:---------------------------------------|--------------:|
| Burford,M. O.  | M. O.                          | Burford   | Not Found           |               |              |             |         |                |           |          |               |       0 |                                        |             0 |
| Comendant,T.   | T.                             | Comendant | Not Found           |               |              |             |         |                |           |          |               |       0 |                                        |             0 |
| Gaertner,L. M. | L. M.                          | Gaertner  | Not Found           |               |              |             |         |                |           |          |               |       0 |                                        |             0 |
| Gehlbach,S. J. | S. J.                          | Gehlbach  | Not Found           |               |              |             |         |                |           |          |               |       0 |                                        |             0 |
| Grenier,E. M.  | E. M.                          | Grenier   | Not Found           |               |              |             |         |                |           |          |               |       0 |                                        |             0 |
| Linville,R.    | R.                             | Linville  | Not Found           |               |              |             |         |                |           |          |               |       0 |                                        |             0 |
| Rollin,S. J.   | S. J.                          | Rollin    | Not Found           |               |              |             |         |                |           |          |               |       0 |                                        |             0 |
| Hahn,D. R.     | Phillip D.                     | Hahn      | 0000-0002-7666-7632 | 0.0           | 2.0          | 2.0         | 13.0    | 0.0            | 0.0       |          |               |      15 | Pediatrics                             |             1 |
| Kalish,S.      | Sue Ann S.                     | Kalish    | 0000-0002-9677-0602 | 0.0           | 0.0          | 0.0         | 0.0     | 0.0            | 0.0       |          |               |       0 |                                        |             1 |
| Boyd,E.        | Alicia E.                      | Boyd      | 0000-0002-2077-3823 | 0.0           | 0.0          | 0.0         | 0.0     | 0.0            | 0.0       |          |               |       0 |                                        |             3 |
| Boyd,E.        | E. Fidelma                     | Boyd      | 0000-0001-7435-4897 | 0.0           | 1.0          | 2.0         | 130.0   | 0.0            | 0.0       |          |               |     131 | Applied and Environmental Microbiology |             3 |
| Boyd,E.        | Ethan, Ethan Robert, E. Robert | Boyd      | 0009-0002-1269-7186 | 0.0           | 3.0          | 0.0         | 1.0     | 0.0            | 0.0       |          |               |       4 | Acta Anaesthesiologica Scandinavica    |             3 |
| Pearson,J. A.  | J. Stephen                     | Pearson   | 0000-0002-6252-2432 | 0.0           | 0.0          | 0.0         | 0.0     | 0.0            | 0.0       |          |               |       0 |                                        |             3 |
| Pearson,J. A.  | J Michael                      | Pearson   | 0000-0002-1000-4107 | 0.0           | 1.0          | 1.0         | 0.0     | 0.0            | 0.0       |          |               |       1 |                                        |             3 |
| Pearson,J. A.  | Scott J.                       | Pearson   | 0000-0003-3676-2050 | 0.0           | 1.0          | 0.0         | 0.0     | 0.0            | 0.0       |          |               |       1 |                                        |             3 |

In [41]:
author_names = studyYear_df['Authors, Primary'].unique().tolist()
author_widget = widgets.Dropdown(
    options=author_names,
    description='Select Author:',
    style={'description_width': '100px'}
)
display(author_widget)

def on_author_change(change):
    global selectedAuthor
    selectedAuthor = change['new']

author_widget.observe(on_author_change, names='value')
selectedAuthor = author_widget.value


Dropdown(description='Select Author:', options=('Boyd,E.', 'Burford,M. O.', 'Comendant,T.', 'Gaertner,L. M.', …

In [42]:
element_l = ['email','education', 'employment', 'works','funding']

input = selectedAuthor
orcid_l = list(authorORCIDMetadata_df[authorORCIDMetadata_df['input']==input]['orcid'].values)

if orcid_l[0] == 'Not Found':
    print(f'No ORCID was found for {input} from the {studyYear} class')
else:
    if len(orcid_l) == 1:
        display(Markdown(f'{input} from the {studyYear} class has 1 ORCID match: [{orcid_l[0]}](https://commons.datacite.org/orcid.org/{orcid_l[0]})'))
    else:
        print(f'{input} from the {studyYear} class has {len(orcid_l)} ORCID matchs: ', ', '.join(orcid_l))
    for element in element_l:
        data = orcid_dfs[element]
        df = data[data['orcid'].isin(orcid_l)]
        if len(df) > 0:
            df = df.fillna('')
            display(Markdown(f'**{element}: ({len(df)})**'))
            display(Markdown(tabulate(df, headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))
        else:
            display(Markdown(f'{input} from the {studyYear} class has no {element} metadata.'))

Kalish,S. from the 1994 class has 1 ORCID match: [0000-0002-9677-0602](https://commons.datacite.org/orcid.org/0000-0002-9677-0602)

Kalish,S. from the 1994 class has no email metadata.

**education: (1)**

| input     | given      | family   | orcid               | name   | identifier   | department   | degree   | start   | end   |
|:----------|:-----------|:---------|:--------------------|:-------|:-------------|:-------------|:---------|:--------|:------|
| Kalish,S. | Sue Ann S. | Kalish   | 0000-0002-9677-0602 |        |              |              |          |         |       |

**employment: (1)**

| input     | given      | family   | orcid               | name   | identifier   | start   | end   |
|:----------|:-----------|:---------|:--------------------|:-------|:-------------|:--------|:------|
| Kalish,S. | Sue Ann S. | Kalish   | 0000-0002-9677-0602 |        |              |         |       |

**works: (1)**

| input     | given      | family   | orcid               | source   | title   | journal   | publicationDate   | identifier   |
|:----------|:-----------|:---------|:--------------------|:---------|:--------|:----------|:------------------|:-------------|
| Kalish,S. | Sue Ann S. | Kalish   | 0000-0002-9677-0602 |          |         |           |                   |              |

**funding: (1)**

| input     | given      | family   | orcid               | title   | funder   | identifier   | award   | start   | end   |
|:----------|:-----------|:---------|:--------------------|:--------|:---------|:-------------|:--------|:--------|:------|
| Kalish,S. | Sue Ann S. | Kalish   | 0000-0002-9677-0602 |         |          |              |         |         |       |

In [ ]:
oneMatch_df = counts_df[counts_df['matchCounts'] == 1]

In [ ]:
#oneMatch_Counts_df = oneMatch_df.merge(counts_df, on='input', how='left')
#oneMatch_Counts_df = oneMatch_Counts_df.fillna('')

In [ ]:
display(Markdown(f'**{len(oneMatch_df)} People with one ORCID match**'))
display(Markdown(tabulate(oneMatch_df.sort_values(by='input'), headers=list(oneMatch_df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))


In [ ]:
element_l = ['email','education', 'employment', 'works','funding']
input = 'Durst,Paul'
orcid = oneMatch_df[oneMatch_df['input']==input]['orcid'].values[0]
print(f'{input} has one ORCID match: {orcid}')
for element in element_l:
    data = orcid_dfs[element]
    df = data[data['orcid'] == orcid]
    if len(df) > 0:
        df = df.fillna('')
        display(Markdown(f'**{element}: ({len(df)})**'))
        display(Markdown(tabulate(df, headers=list(df.columns), tablefmt='pipe', floatfmt='.0f', showindex=False)))
    else:
        display(Markdown(f'{input} has no {element} metadata.'))

In [ ]:
author_df.to_csv('/Users/metadatagamechanger/ProjectsAndPlans/FieldStations/MooreaStudentPapers/studentORCIDs/matchedNames.csv', index=False)